# Teks — Augmentasi Data Teks (data latih sedikit)
**Kapan dipakai:** data berlabel hanya ratusan baris. Augmentasi = membuat salinan teks yang
sedikit diubah agar model melihat variasi lebih banyak:
- **hapus** kata acak, **tukar** posisi 2 kata acak;
- **slang**: ganti kata baku dengan versi alay (kamus alay dibalik: baku → slang).

**Aturan penting:** augmentasi HANYA pada data latih; validasi dan test tetap teks asli.

**Data demo:** EmoT (emosi tweet), sengaja hanya dipakai 400 baris (sel Baca data).

In [1]:
import random
import re

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/text/classification"   # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/emot_train.csv"
TEST_CSV = f"{DATA_FOLDER}/emot_test.csv"
TEST_KEY_CSV = f"{DATA_FOLDER}/emot_test_labels.csv"   # hanya ada di data latihan
SLANG_CSV = "../../../data_latihan/preprocessing/kamus_alay_salsabila.csv"

TEXT_COL = "text"
TARGET = "label"
N_COPIES = 2          # jumlah teks augmentasi per teks asli
CHANGE_CHANCE = 0.15  # peluang tiap kata diubah
SEED = 42
random.seed(SEED)

## Baca data

In [3]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
train = train.sample(400, random_state=SEED)   # HAPUS baris ini saat lomba (simulasi data sedikit)
print("train:", train.shape, "| test:", test.shape)
print(train[TARGET].value_counts())
train.head()

train: (400, 3) | test: (440, 2)
label
anger      105
happy       96
sadness     92
fear        57
love        50
Name: count, dtype: int64


,id,label,text
2458,2459,sadness,Dukaku karna tak mampu katakan sedih ku karna ...
70,71,anger,Jadi wanita jangan suka menghancurkan hubungan...
358,359,sadness,Ini dia si pembuat sembab berpasang-pasang mat...
325,326,love,"Suka sama cewek beda kelas dong, gue di kelas ..."
2307,2308,sadness,Kabar baik ku pagi ini... dpt tagihan uang keg...


## Bersihkan teks

In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[username\]|http\S+|@\w+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())


print(clean_text("[USERNAME] Seneng bgt hari ini!!! http://x.id"))

train[TEXT_COL] = train[TEXT_COL].fillna("").apply(clean_text)
test[TEXT_COL] = test[TEXT_COL].fillna("").apply(clean_text)

seneng bgt hari ini


## Split train / validasi

In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    train[TEXT_COL], train[TARGET], test_size=0.25, stratify=train[TARGET], random_state=SEED
)

## Kamus terbalik: kata baku → daftar versi slang

In [6]:
slang_table = pd.read_csv(SLANG_CSV)
standard_to_slang = {}
for slang, standard in zip(slang_table["slang"], slang_table["baku"]):
    if " " in str(standard):
        continue                       # lewati padanan lebih dari satu kata
    if standard not in standard_to_slang:
        standard_to_slang[standard] = []
    standard_to_slang[standard].append(slang)
print(len(standard_to_slang), "kata baku punya versi slang")

1917 kata baku punya versi slang


## Fungsi augmentasi satu teks
1. Hapus kata acak. 2. Tukar posisi 2 kata acak. 3. Ganti kata baku dengan slang.

In [7]:
def augment(text):
    words = text.split()
    if len(words) < 3:
        return text
    kept = []
    for word in words:
        if random.random() > CHANGE_CHANCE:
            kept.append(word)
    if len(kept) >= 2:
        first, second = random.sample(range(len(kept)), 2)
        kept[first], kept[second] = kept[second], kept[first]
    result = []
    for word in kept:
        if word in standard_to_slang and random.random() < CHANGE_CHANCE * 2:
            word = random.choice(standard_to_slang[word])
        result.append(word)
    return " ".join(result)


print(X_train.iloc[0])
print(augment(X_train.iloc[0]))

jangan pilih jokowi atau golput aja sekalian nanti pancasila yg menjadi dasar negara kita sekarang akan ikutan terganti mau mmmmmoh
jangan jokowi ikutan golput aja sekalian pancasila mendj dsr kitaa sekarang akan atau terganti mau


## Tambahkan salinan augmentasi ke data latih
Tiap teks asli mendapat `N_COPIES` salinan dengan label yang sama.

In [8]:
def add_augmented(texts, labels):
    new_texts = []
    new_labels = []
    for text, label in zip(texts, labels):
        for copy_number in range(N_COPIES):
            new_texts.append(augment(text))
            new_labels.append(label)
    all_texts = pd.concat([texts, pd.Series(new_texts)], ignore_index=True)
    all_labels = pd.concat([labels, pd.Series(new_labels)], ignore_index=True)
    return all_texts, all_labels


X_augmented, y_augmented = add_augmented(X_train, y_train)
print("data latih:", len(X_train), "→ setelah augmentasi:", len(X_augmented))

data latih: 300 → setelah augmentasi: 900


## Model
Pilih SATU vectorizer dan SATU model: hapus `#` di baris yang ingin dipakai.

In [9]:
vectorizer = make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
                        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2))
# vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)

model = LogisticRegression(C=5, max_iter=3000)
# model = LinearSVC(C=0.5)
# model = ComplementNB(alpha=0.3)

pipe = make_pipeline(vectorizer, model)

## Evaluasi: tanpa vs dengan augmentasi (validasi = teks asli)

In [10]:
pipe.fit(X_train, y_train)
plain_predictions = pipe.predict(X_val)
print("F1 macro tanpa augmentasi :", round(f1_score(y_val, plain_predictions, average="macro"), 4))

pipe.fit(X_augmented, y_augmented)
val_predictions = pipe.predict(X_val)
print("F1 macro dengan augmentasi:", round(f1_score(y_val, val_predictions, average="macro"), 4))
print(classification_report(y_val, val_predictions))

F1 macro tanpa augmentasi : 0.4909


F1 macro dengan augmentasi: 0.5046
              precision    recall  f1-score   support

       anger       0.59      0.65      0.62        26
        fear       0.44      0.29      0.35        14
       happy       0.64      0.58      0.61        24
        love       0.70      0.54      0.61        13
     sadness       0.30      0.39      0.34        23

    accuracy                           0.51       100
   macro avg       0.53      0.49      0.50       100
weighted avg       0.53      0.51      0.51       100



## Latih ulang di semua data (+ augmentasi) + submission
Augmentasi tidak membantu di validasi? Latih dengan `train[TEXT_COL], train[TARGET]` saja.

In [11]:
X_all, y_all = add_augmented(train[TEXT_COL], train[TARGET])
pipe.fit(X_all, y_all)
predictions = pipe.predict(test[TEXT_COL])

submission = pd.DataFrame()
submission["id"] = test["id"]
submission["label"] = predictions
submission.to_csv("submission.csv", index=False)
submission.head()

,id,label
0,100001,anger
1,100002,sadness
2,100003,anger
3,100004,happy
4,100005,happy


## Cek dengan kunci jawaban latihan
**HAPUS sel ini saat lomba**: file kunci hanya ada di data latihan.

In [12]:
test_key = pd.read_csv(TEST_KEY_CSV)["label"]
print("F1 macro test (kunci latihan):", round(f1_score(test_key, predictions, average="macro"), 4))

F1 macro test (kunci latihan): 0.5096
